# Input Parameters

In [ ]:
%sql
CREATE WIDGET TEXT catalog_source DEFAULT 'bronze_dev';
CREATE WIDGET TEXT catalog_sink DEFAULT 'silver_dev';

DECLARE OR REPLACE VARIABLE schema_source STRING DEFAULT 'wanderbricks';
DECLARE OR REPLACE VARIABLE schema_sink STRING DEFAULT 'reservations';

# Create Schema Sink

In [ ]:
%sql
CREATE SCHEMA IF NOT EXISTS IDENTIFIER(:catalog_sink || '.' || schema_sink)

# Load tables

## Users

In [ ]:
%sql
CREATE OR REPLACE TABLE IDENTIFIER(:catalog_sink || '.' || schema_sink || '.users') CLUSTER BY (user_id)
COMMENT 'Usuários do Wanderbricks enriquecidos com dados geográficos (continente via join com countries). Inclui segmentação por user_type e is_business para análises de base de clientes.'
TBLPROPERTIES (
  'data.quality.level' = 'Silver',
  'delta.autoOptimize.optimizeWrite' = 'true',
  'delta.autoOptimize.autoCompact' = 'true'
)
AS
SELECT
  u.user_id,
  u.email,
  u.name,
  u.country,
  c.continent,
  u.user_type,
  u.created_at,
  u.is_business,
  u.company_name, 
  convert_timezone('-03:00', current_timestamp()) AS processing_timestamp,
  'overwrite' AS process_write_mode
FROM
  IDENTIFIER(:catalog_source || '.' || schema_source || '.users') AS u
  JOIN IDENTIFIER(:catalog_source || '.' || schema_source || '.countries') AS c
    ON u.country = c.country

## Bookings

In [ ]:
%sql
CREATE OR REPLACE TABLE IDENTIFIER(:catalog_sink || '.' || schema_sink || '.bookings') CLUSTER BY (booking_id)
COMMENT 'Reservas de imóveis com datas, valores e status. Relaciona users, properties e payments. Base para análises de ocupação e receita.'
TBLPROPERTIES (
  'data.quality.level' = 'Silver',
  'delta.autoOptimize.optimizeWrite' = 'true',
  'delta.autoOptimize.autoCompact' = 'true'
)
AS
SELECT
  booking_id,
  user_id,
  property_id,
  check_in,
  check_out,
  guests_count,
  total_amount,
  status,
  created_at,
  updated_at,
  convert_timezone('-03:00', current_timestamp()) AS processing_timestamp,
  'overwrite' AS process_write_mode
FROM
  IDENTIFIER(:catalog_source || '.' || schema_source || '.bookings')

## Payments

In [ ]:
%sql
CREATE OR REPLACE TABLE IDENTIFIER(:catalog_sink || '.' || schema_sink || '.payments') CLUSTER BY (payment_id)
COMMENT 'Transações financeiras das reservas com métodos de pagamento e status. Relaciona-se com bookings para análises financeiras e faturamento.'
TBLPROPERTIES (
  'data.quality.level' = 'Silver',
  'delta.autoOptimize.optimizeWrite' = 'true',
  'delta.autoOptimize.autoCompact' = 'true'
)
AS
SELECT
  payment_id,
  booking_id,
  amount,
  payment_method,
  status,
  payment_date,
  convert_timezone('-03:00', current_timestamp()) AS processing_timestamp,
  'overwrite' AS process_write_mode
FROM
  IDENTIFIER(:catalog_source || '.' || schema_source || '.payments')